In [1]:
from datetime import datetime
import glob
import shutil
import time

import win32com.client  # Windows COM 인터페이스 사용
import xml.etree.ElementTree as ET
import pandas as pd

import os.path

# 입력 파라미터
###################################################################################################################
network_path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입"
network_file_name = r"화성~서울(140-유고지점)_260526.inpx"
network_full_path = os.path.join(network_path, network_file_name)

copy_network_path = os.path.join(network_path, f"화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615.inpx")

csv_path = r"C:\Users\alswl\Desktop\유고 시나리오\추정용 시나리오"
csv_file_name = r"synthetic_scenarios_all_140_with_set_type(영향권2(진입))_3차로폐쇄-민진.csv"

# 유고 발생 시각
acc_start_time = 3600
###################################################################################################################
# =========================================================
# .mer → parquet 변환 함수
# =========================================================
def convert_mer_to_parquet(mer_file, delete_original=True):
    try:
        print(f"[변환 시작] {mer_file}")

        with open(mer_file, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        data_start_idx = None
        for i, line in enumerate(lines):
            if "Measurem." in line:
                data_start_idx = i
                break

        if data_start_idx is None:
            print(f"[건너뜀] 데이터 헤더를 찾지 못함: {mer_file}")
            return

        df = pd.read_csv(
            mer_file,
            sep=";",
            skiprows=data_start_idx,
            encoding="utf-8",
            engine="python"
        )

        df.columns = [str(col).strip() for col in df.columns]

        parquet_path = mer_file.replace(".mer", ".parquet")

        df.to_parquet(
            parquet_path,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"[저장 완료] {parquet_path}")

        if delete_original:
            os.remove(mer_file)
            print(f"[원본 삭제 완료] {mer_file}")

    except Exception as e:
        print(f"[변환 실패] {mer_file}")
        print(e)


csv_full_path  = os.path.join(csv_path, csv_file_name)

# 시나리오 파일 읽기
senario_df = pd.read_csv(csv_full_path, encoding="cp949")

Vissim = win32com.client.Dispatch("Vissim.Vissim")  # Vissim 인스턴스 생성

for index, row in senario_df.iterrows():

    # 교통량
    Q_main_in_vehph = row["base_main_in_vph"]
    Q_ramp_in_per_ramp = row["q_ramp_in_vph_per"]

    # 램프 유출 비율
    Q1_ramp_out_vehph = row["q1_ramp_out_vph_per_ratio"]
    Q2_ramp_out_vehph = row["q2_ramp_out_vph_per_ratio"]
    Q3_ramp_out_vehph = row["q3_ramp_out_vph_per_ratio"]

    # 본선 유입 중차량 비율
    V1_main_in_ratio = row["V1_main_in_ratio"]
    V2_main_in_ratio = row["V2_main_in_ratio"]
    V3_main_in_ratio = row["V3_main_in_ratio"]
    V4_main_in_ratio = row["V4_main_in_ratio"]
    V5_main_in_ratio = max(row["V5_main_in_ratio"], 0.001)

    # 램프 유입 중차량 비율
    V1_ramp_in_ratio = row["V1_ramp_in_ratio"]
    V2_ramp_in_ratio = row["V2_ramp_in_ratio"]
    V3_ramp_in_ratio = row["V3_ramp_in_ratio"]
    V4_ramp_in_ratio = row["V4_ramp_in_ratio"]
    V5_ramp_in_ratio = max(row["V5_ramp_in_ratio"], 0.001)

    # 종단경사
    #grade_percent = row["grade_percent"] * 0.01

    # 유고 지속시간
    incident_duration_min = row["incident_duration_min"]

    # 유고 차로수
    lane_closure_count = int(row["lane_closure_count"])

    # 랜덤시드
    random_seed = int(row["random_seed"])

    # 유고 발생 지점

    incident_location = row["location_name"]
    locations = [x.strip() for x in incident_location.split(",")] # [본선1, 본선2]
    location = locations[0]

    shutil.copy(network_full_path, copy_network_path)

    # XML 로드
    tree = ET.parse(copy_network_path)
    root = tree.getroot()


    # 본선, 유입램프 교통량
    for vehicle_input in root.findall(".//vehicleInput"):
        input_name = vehicle_input.get("name")
        vehicleInput = vehicle_input.find("timeIntVehVols")

        if vehicleInput is None:
            continue
        for vol in vehicleInput.findall("timeIntervalVehVolume"):
            if input_name == "main":
                vol.set("volume", str(Q_main_in_vehph))
            elif input_name in ["ramp1-1", "ramp1-2"]:
                vol.set("volume", str(Q_ramp_in_per_ramp))

    # 본선/ 램프 유입 중차량 비율
    for vehicle_comp in root.findall(".//vehicleComposition"):
        if vehicle_comp.get("name") == "main":
            relflow_100 = V1_main_in_ratio
            relflow_300 = V2_main_in_ratio
            relflow_630 = V3_main_in_ratio
            relflow_640 = V4_main_in_ratio
            relflow_650 = V5_main_in_ratio

        elif vehicle_comp.get("name") == "ramp":
            relflow_100 = V1_ramp_in_ratio
            relflow_300 = V2_ramp_in_ratio
            relflow_630 = V3_ramp_in_ratio
            relflow_640 = V4_ramp_in_ratio
            relflow_650 = V5_ramp_in_ratio
        else:
            continue
        # <vehCompRelFlows> 태그 찾기
        vehCompRelFlows = vehicle_comp.find("vehCompRelFlows")
        if vehCompRelFlows is not None:
            # <vehicleCompositionRelativeFlow> 태그 찾기
            for relflow in vehCompRelFlows.findall("vehicleCompositionRelativeFlow"):
                vehType = relflow.get("vehType")
                if vehType == "100": # 승용차
                    relflow.set("relFlow", str(relflow_100))
                elif vehType == "300": # 버스
                    relflow.set("relFlow", str(relflow_300))
                elif vehType == "630": # 소형화물
                    relflow.set("relFlow", str(relflow_630))
                elif vehType == "640": # 중형화물
                    relflow.set("relFlow", str(relflow_640))
                elif vehType == "650": # 대형화물
                    relflow.set("relFlow", str(relflow_650))

    # 램프 유출 비율
    for routing_decision  in root.findall(".//vehicleRoutingDecisionStatic"):
        decision_no = routing_decision.get("no")

        # decision별 Q값 매핑
        if decision_no == "1":
            Q = Q1_ramp_out_vehph
        elif decision_no == "2":
            Q = Q2_ramp_out_vehph
        elif decision_no == "3":
            Q = Q3_ramp_out_vehph
        else:
            continue

        veh_routes = routing_decision.find("vehRoutSta")
        if veh_routes is None:
            continue

        for route in veh_routes.findall("vehicleRouteStatic"):
            route_no = route.get("no")

            if route_no == "1":  # 직진
                route.set("relFlow", f"2 0:{round(1 - Q, 3)}")

            elif route_no == "2":  # 진출
                route.set("relFlow", f"2 0:{round(Q, 3)}")


    # 유고 발생 위치 조정
    for rsa in root.findall(".//reducedSpeedArea"):
        name = rsa.get("name")
        if name in locations:
            # 유고 지속시간 설정
            rsa.set("timeFrom", str(acc_start_time))
            rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))


    # 랜덤시드 설정
    for rds in root.findall(".//simulation"):
        rds.set("randSeed", str(random_seed))


    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"입력값 | main={Q_main_in_vehph}, ramp={Q_ramp_in_per_ramp}, seed={random_seed}"
    )


    tree.write(copy_network_path, encoding="utf-8", xml_declaration=True)
    before_mer_files = set(glob.glob(os.path.join(network_path, "*.mer")))

    # VISSIM 네트워크 로드
    Vissim.LoadNet(copy_network_path)
    Vissim.Graphics.CurrentNetworkWindow.SetAttValue('QuickMode', 1) # 퀵 모드 활성화

    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        "VISSIM 시뮬레이션 실행 중..."
    )
    Vissim.Simulation.RunContinuous()
    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        "VISSIM 시뮬레이션 실행 완료"
    )

    # 파일 저장 완료 대기
    time.sleep(0.5)

    # 시뮬레이션 후 새로 생성된 mer만 변환
    after_mer_files = set(glob.glob(os.path.join(network_path, "*.mer")))
    new_mer_files = sorted(list(after_mer_files - before_mer_files))

    if len(new_mer_files) == 0:
        print("[주의] 새로 생성된 .mer 파일을 찾지 못했습니다.")
    else:
        for mer_file in new_mer_files:
            convert_mer_to_parquet(mer_file, delete_original=True)

Vissim.Simulation.Stop()
Vissim.Exit()
print("VISSIM 시뮬레이션 종료")
print("==============================================================================================================")



[2026-06-17 18:20:22] 입력값 | main=3906, ramp=227, seed=422181
[2026-06-17 18:20:23] VISSIM 시뮬레이션 실행 중...
[2026-06-17 18:38:50] VISSIM 시뮬레이션 실행 완료
[변환 시작] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_328.mer
[저장 완료] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_328.parquet
[원본 삭제 완료] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_328.mer
[변환 시작] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_329.mer
[저장 완료] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_329.parquet
[원본 삭제 완료] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권2_진입\화성~서울(110-영향권2_진입_3차로폐쇄-추정용)_260615_329.mer
[변환 시작] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Rando